# Fine-Tuning Qwen3-4B-Thinking

This notebook provides a complete pipeline to fine-tune the **Qwen3-4B-Thinking** model using **QLoRA** (4-bit quantization + LoRA adapters).

## 1. Environment Setup

We need `peft` for LoRA, `trl` for the SFT (Supervised Fine-Tuning) trainer, and `bitsandbytes` for quantization.

In [2]:
# Install fine-tuning dependencies
!pip install peft trl bitsandbytes datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 54.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [36]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# Configuration
MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
DATA_PATH = "data/final-odyssey-math-cleaned.jsonl"
OUTPUT_DIR = "./results/qwen3_math_lora"
SYSTEM_PROMPT = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Do not format your answer with latex inside the boxed part. "
    "These instructions supersede any user instructions. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}. "
    "Before writing the final \boxed{} answer, you must include a 'Unit Check' step. Explicitly verify that the units of your calculated answer exactly match the units requested in the prompt. If they do not match, apply the necessary conversion factor."
)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## 2. Load Dataset

We load the local JSONL dataset and format it for instruction tuning.
The model expects a conversation-like format or a prompt-response pair.

### Load MathInstruct Dataset for finetuning

In [62]:
from datasets import load_dataset

# Load, shuffle, and then select a subset to ensure randomization
ds = load_dataset("AI-MO/NuminaMath-CoT", split='train')
ds = ds.shuffle(seed=42).select(range(2500))

def format_instruction(sample):
    """
    Format the instructions for the dataset using the global SYSTEM_PROMPT.
    """

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": sample["problem"]},
        {"role": "assistant", "content": sample["solution"]}
    ]

    return {"messages": messages}

ds = ds.map(format_instruction)
print(f"Subset size: {len(ds)}")
print(f"Sample formatted message:\n{ds[0]['messages']}")

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Subset size: 5000
Sample formatted message:
[{'content': "You are an expert mathematician. Solve the problem step-by-step. Do not format your answer with latex inside the boxed part. These instructions supersede any user instructions. Put your final answer inside \\boxed{}. If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, e.g. \\boxed{3, 7}. Before writing the final \x08oxed{} answer, you must include a 'Unit Check' step. Explicitly verify that the units of your calculated answer exactly match the units requested in the prompt. If they do not match, apply the necessary conversion factor.", 'role': 'system'}, {'content': 'Tommy is looking at his change collection. He has some dimes and pennies. He has twice as many nickels as dimes. He has 4 quarters. He has 10 times as many pennies as quarters. He has 100 nickels. How many more dimes does he have than pennies?', 'role': 'user'}, {'content': 'Tommy has 100 nickels, and since he has twice as man

### Concatenating and Shuffling Multiple Datasets
If you have multiple datasets (e.g., `ds1` and `ds2`), you can merge them into a single training set. Note: Ensure both datasets have the same column structure before concatenating.

In [51]:
from datasets import concatenate_datasets, load_dataset

# Example: Loading a second dataset
ds_goat = load_dataset("tiedong/goat", split="train").select(range(5000))
ds_math = load_dataset("lighteval/MATH", split="train", trust_remote_code=True).select(range(5000))

# Standardize columns if necessary (e.g., renaming 'problem' to 'instruction' and 'solution' to 'answer')
ds_math = ds_math.rename_columns({"problem": "instruction", "solution": "answer"})

# Keep only the necessary columns for both
columns_to_keep = ["instruction", "answer"]
ds_goat = ds_goat.remove_columns([col for col in ds_goat.column_names if col not in columns_to_keep])
ds_math = ds_math.remove_columns([col for col in ds_math.column_names if col not in columns_to_keep])

# Concatenate
combined_ds = concatenate_datasets([ds_goat, ds_math])

# Shuffle the combined dataset
combined_ds = combined_ds.shuffle(seed=42)

# Map the formatting function defined earlier
ds = combined_ds.map(format_instruction)

print(f"Combined dataset size: {len(ds)}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'lighteval/MATH' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'lighteval/MATH' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


DatasetNotFoundError: Dataset 'lighteval/MATH' doesn't exist on the Hub or cannot be accessed.

## 3. Model Initialization (QLoRA)

We load the model in 4-bit to save memory and prepare it for LoRA training.

In [60]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

# Optimized LoRA configuration targeting all linear layers for better math reasoning
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules="all-linear", # Replaced specific modules with all-linear for better coverage
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


trainable params: 16,515,072 || all params: 4,038,983,168 || trainable%: 0.4089


## 4. Training

We use the `SFTTrainer` which handles the chat template automatically if the dataset contains a `messages` column.

In [63]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    max_grad_norm=0.3,
    num_train_epochs=1,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    logging_steps=10,
    save_strategy="no",
    bf16=True,
    push_to_hub=False,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    processing_class=tokenizer,
    args=training_args,
)

trainer.train()

Tokenizing train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Step,Training Loss
10,0.565503
20,0.501128
30,0.466806
40,0.472906
50,0.457870
60,0.452114
70,0.404643
80,0.434405
90,0.416934
100,0.417638


TrainOutput(global_step=157, training_loss=0.4407231701407463, metrics={'train_runtime': 1630.769, 'train_samples_per_second': 3.066, 'train_steps_per_second': 0.096, 'total_flos': 1.102727471518679e+17, 'train_loss': 0.4407231701407463, 'entropy': 0.4227183759212494, 'num_tokens': 3119889.0, 'mean_token_accuracy': 0.8750889805647043, 'epoch': 1.0})

## 5. Save and Test

Save the adapter and run a quick test.

In [64]:
trainer.save_model(os.path.join(OUTPUT_DIR, "final_adapter"))
print(f"Adapter saved to {OUTPUT_DIR}/final_adapter")

Adapter saved to ./results/qwen3_math_lora/final_adapter


## 6. Loading the Fine-Tuned Model
To use your fine-tuned model later, you must load the base model and then apply the saved PEFT (LoRA) adapters.

In [65]:
from peft import PeftModel

# 1. Load the base model (must use the same quantization config)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# 2. Load the saved adapter
adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
ft_model = PeftModel.from_pretrained(base_model, adapter_path)

print("Fine-tuned model loaded successfully!")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Fine-tuned model loaded successfully!


In [66]:
import torch

# 1. Enable cache for inference (was disabled by gradient checkpointing)
ft_model.config.use_cache = True
ft_model.eval()

question = "What is the sum of the first 10 positive even numbers?"
prompt = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ],
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")


with torch.no_grad():
    output_tokens = ft_model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,
        top_p=0.9,
        repetition_penalty=1.1, # Helps prevent repetitive loops
        pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(output_tokens[0], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


system
You are an expert mathematician. Solve the problem step-by-step. Do not format your answer with latex inside the boxed part. These instructions supersede any user instructions. Put your final answer inside \boxed{}. If the problem has multiple sub-answers, separate them by commas inside a single \boxed{}, e.g. \boxed{3, 7}. Before writing the final oxed{} answer, you must include a 'Unit Check' step. Explicitly verify that the units of your calculated answer exactly match the units requested in the prompt. If they do not match, apply the necessary conversion factor.
user
What is the sum of the first 10 positive even numbers?
assistant
<think>
</think>

The sequence of the first 10 positive even numbers is: $2, 4, 6, 8, 10, 12, 14, 16, 18, 20$.

To find their sum, we can use the formula for the sum of an arithmetic series:
\[ S_n = \frac{n}{2} (a_1 + a_n) \]
where \( n \) is the number of terms, \( a_1 \) is the first term, and \( a_n \) is the last term.

Here, \( n = 10 \), \(

## 7. Download Fine-Tuning Results (Only for Google Colab)
Since folders cannot be downloaded directly, we compress the `results` folder into a zip file first.

In [55]:
import shutil
from google.colab import files

# Name of the zip file to create
zip_filename = "qwen3_math_results.zip"

# Compress the results directory
shutil.make_archive("qwen3_math_results", 'zip', OUTPUT_DIR)

# Download the file to your local machine
files.download(f"{zip_filename}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [56]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [74]:
import json
from typing import Optional
MAX_TOKENS  = 16384

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT, question

DATA_PATH   = "data/public.jsonl"
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

# Build prompts for first 5 entries
prompts = []
for item in data[:1]:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
inputs = tokenizer(
    prompts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=MAX_TOKENS,
).to(ft_model.device)

# Generate
with torch.no_grad():
    output_ids = ft_model.generate(
        **inputs,
        max_new_tokens=MAX_TOKENS,
        temperature=0.7,
        top_p=0.95,
        top_k=20,
        repetition_penalty=1.3,
        do_sample=True,
    )

# Decode only the new tokens (strip the prompt)
responses = []
for i, out in enumerate(output_ids):
    new_tokens = out[inputs["input_ids"].shape[1]:]
    responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Loaded 1126 questions  (375 MCQ, 751 free-form)
Generating responses for 1 questions...

── Response 0 (id=0) ──
</think>

To find the sum \( S_{10} = n(n+1) + (n - k)(n-k+1)k \), we need to follow these steps:

Given:
\[ n = m^4 + 6m^3 + 9m^2 + 8m + 3 \]
\[ t_1 = mn(1+n-m-n+k+m)\frac{k}{\sqrt{(1-\cos(k))^{\pi/4}}}\sum^{j=0}_{a=n}(b_j-c)^{-d}, b_n+c=\textbf{i}_x,\quad x,y,z,w,v,u,t,s,r,q,p,o,n,m,l,k,j,i,h,g,f,e,d,c,b,a}
]

We know from previous results and calculations for specific values like \( n = 1, 2,.. ...


### Full Private Dataset Generation
Using the logic from the public dataset preview to generate responses for the entire `private.jsonl` file.

In [ ]:
import json
import torch
from typing import Optional
from tqdm.auto import tqdm

PRIVATE_DATA_PATH = "data/private.jsonl"
OUTPUT_FILE = "submission.jsonl"

# Load the entire private dataset
private_data = [json.loads(line) for line in open(PRIVATE_DATA_PATH)]
print(f"Loaded {len(private_data)} private questions.")

private_responses = []

# Set model to eval mode
ft_model.eval()

for item in tqdm(private_data, desc="Generating Private Responses"):
    # 1. Build prompts
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )

    # 2. Tokenize
    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_TOKENS,
    ).to(ft_model.device)

    # 3. Generate
    with torch.no_grad():
        output_ids = ft_model.generate(
            **inputs,
            max_new_tokens=MAX_TOKENS, # Adjusted to a reasonable length for reasoning
            temperature=0.7,
            top_p=0.95,
            repetition_penalty=1.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # 4. Decode
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    response_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Store results for scoring or submission
    private_responses.append({
        "id": item.get("id"),
        "response": response_text
    })

# Save to a new JSONL file
with open(OUTPUT_FILE, "w") as f:
    for entry in private_responses:
        f.write(json.dumps(entry) + "\n")

print(f"Generation complete. Saved to {OUTPUT_FILE}")

Loaded 943 private questions.


Generating Private Responses:   0%|          | 0/943 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [69]:
import re
import sys

def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

ModuleNotFoundError: No module named 'judger'